# 05 — Speed-conditioned VAE (comparison model M4)

Same network as notebook 04, but **the walking speed is given to the model as an extra input** (a *conditional* VAE):

```
encoder:  window (60×134) + speed  → μ, log σ² (64) → z
decoder:  z + speed                → window (60×134)
```

Because speed is a separate input, you can **ask for a speed**: encode a walk, then decode it with a different speed.
This is the "model built for speed control" that your post-hoc speed direction (notebook 04) is compared against.

**What the notebook measures (on test people):**
1. Reconstruction with the true speed (same numbers as notebook 04, for a fair comparison)
2. **Speed control:** decode each test walk at a range of requested speeds and measure the speed it actually walks at. Is it monotonic? How close to the request?
3. **Does z still contain speed?** A probe z → speed. Ideally low: speed should live in the input, not in z. If high, the decoder may be ignoring the speed input.
4. New walks generated from random z at chosen speeds (stick figures)
5. A side-by-side with the VAE's direction edit, if a notebook-04 run is found

Uses the **full features** (with root speed), because the decoded speed is measured from the root-speed channel.

**Why a speed-control loss?** A plain conditional VAE often *ignores* the speed input: the walk's speed is already in z, so the decoder reads it from there. Then asking for another speed changes almost nothing. To make this a fair "built for speed control" model, each training walk is **also decoded with a random other speed**, and the model is penalised if the decoded root speed doesn't match the requested one (`CONTROL_WEIGHT`). Set it to 0 to get the textbook conditional VAE. The plausibility checks in the Phase-5 notebook will show whether the rest of the body follows along.

## Settings

In [ ]:
from pathlib import Path

PRE_DIR = Path(r"C:\Users\hasna\Documents\Good Practise\CMU\walking\preprocessed")   # output of notebook 02

BETA        = 1.0
LATENT_DIM  = 64
HIDDEN      = 256
ROOT_WEIGHT = 2.0
SPEED_COL   = "fwd_speed_mps"     # condition = measured forward speed of each window
CONTROL_WEIGHT = 100.0  # speed-control loss (see the Model section); 0 = plain conditional VAE.
                        # Higher = better speed control, slightly worse reconstruction.

EPOCHS           = 300
BATCH_SIZE       = 128
LR               = 1e-3
WEIGHT_DECAY     = 1e-4
KL_WARMUP_EPOCHS = 30
PATIENCE         = 30
SEED             = 0

N_TARGET_SPEEDS  = 13     # speeds requested in the control test (spread over the train range)
VAE_RUN          = f"vae_full_z{LATENT_DIM}_b1_s{SEED}"   # notebook-04 run to compare with (optional)

RUN_NAME = f"cvae_full_z{LATENT_DIM}_b{BETA:g}_ctl{CONTROL_WEIGHT:g}_s{SEED}"
OUT_DIR  = PRE_DIR.parent / "models" / RUN_NAME

In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", message=".*[Ii]ll-conditioned.*")

torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| device:", device, "| run:", RUN_NAME)

cfg  = json.loads((PRE_DIR / "config.json").read_text())
norm = np.load(PRE_DIR / "normalization.npz")
FPS, BONES, FEATS = cfg["TARGET_FPS"], cfg["BONES"], cfg["FEATURES"]
MEAN, STD = norm["mean"], norm["std"]
CONTACT = [i for i, f in enumerate(FEATS) if f.startswith("contact_")]
REGULAR = [i for i in range(len(FEATS)) if i not in CONTACT]
ROOT    = [i for i, f in enumerate(FEATS) if f.startswith("root_")]
FWD     = FEATS.index("root_vel_forward")

data = {}
for s in ["train", "val", "test"]:
    d = np.load(PRE_DIR / f"{s}.npz", allow_pickle=True)
    m = pd.read_csv(PRE_DIR / f"{s}_meta.csv")
    data[s] = dict(X=d["X"].astype(np.float32), joints=d["joints"], meta=m, speed=m[SPEED_COL].values.astype(np.float32))
T, D = data["train"]["X"].shape[1:]
assert T % 4 == 0, f"window length {T} must be divisible by 4"

# the speed input is normalised with train statistics
SP_MEAN, SP_STD = float(data["train"]["speed"].mean()), float(data["train"]["speed"].std() + 1e-6)
to_c = lambda v: ((np.asarray(v, np.float32) - SP_MEAN) / SP_STD).astype(np.float32)
for s in data:
    data[s]["c"] = to_c(data[s]["speed"])
    print(f"{s:5s}: {data[s]['X'].shape}  speed {data[s]['speed'].min():.2f}–{data[s]['speed'].max():.2f} m/s")

## Model
Same layers as notebook 04. The speed is added as an extra input channel to the encoder and appended to z before decoding.

**Loss** = reconstruction + β·KL + `CONTROL_WEIGHT` · speed-control loss. The speed-control loss is the squared difference between the decoded forward speed and a randomly requested speed, in normalised units, summed over the frames.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, k=5):
        super().__init__()
        self.net = nn.Sequential(nn.GroupNorm(8, c), nn.GELU(), nn.Conv1d(c, c, k, padding=k // 2),
                                 nn.GroupNorm(8, c), nn.GELU(), nn.Conv1d(c, c, k, padding=k // 2))
    def forward(self, x):
        return x + self.net(x)

class SpeedCVAE(nn.Module):
    def __init__(self, D, T, latent=LATENT_DIM, H=HIDDEN):
        super().__init__()
        self.T4, self.H = T // 4, H
        self.enc = nn.Sequential(
            nn.Conv1d(D + 1, H, 5, padding=2), ResBlock(H),
            nn.Conv1d(H, H, 4, stride=2, padding=1), ResBlock(H),
            nn.Conv1d(H, H, 4, stride=2, padding=1), ResBlock(H),
            nn.Flatten(), nn.Linear(H * self.T4, 2 * latent))
        self.dec_in = nn.Sequential(nn.Linear(latent + 1, H * self.T4))
        self.dec = nn.Sequential(
            ResBlock(H), nn.ConvTranspose1d(H, H, 4, stride=2, padding=1),
            ResBlock(H), nn.ConvTranspose1d(H, H, 4, stride=2, padding=1),
            ResBlock(H), nn.GroupNorm(8, H), nn.GELU(), nn.Conv1d(H, D, 5, padding=2))

    def encode(self, x, c):                    # x: (B, T, D), c: (B,)
        cc = c[:, None, None].expand(-1, x.shape[1], 1)
        mu, logvar = self.enc(torch.cat([x, cc], -1).transpose(1, 2)).chunk(2, dim=-1)
        return mu, logvar.clamp(-10, 10)

    def decode(self, z, c):
        h = self.dec_in(torch.cat([z, c[:, None]], -1)).view(-1, self.H, self.T4)
        return self.dec(h).transpose(1, 2)

    def forward(self, x, c, sample=True):
        mu, logvar = self.encode(x, c)
        z = mu + torch.randn_like(mu) * (0.5 * logvar).exp() if (sample and BETA > 0) else mu
        return self.decode(z, c), mu, logvar

model = SpeedCVAE(D, T).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")

w = torch.ones(D); w[ROOT] = ROOT_WEIGHT
W_REG = w[REGULAR].to(device)
def loss_terms(xh, x, mu, logvar):
    rec = ((xh[..., REGULAR] - x[..., REGULAR]) ** 2 * W_REG).sum((1, 2))
    rec = rec + Fnn.binary_cross_entropy_with_logits(xh[..., CONTACT], x[..., CONTACT], reduction="none").sum((1, 2))
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(-1)
    return rec.mean(), kl.mean()

FWD_MEAN, FWD_STD = float(MEAN[FWD]), float(STD[FWD])
SP_LO, SP_HI = np.percentile(data["train"]["speed"], [2, 98])
def control_loss(z):
    # decode z with random requested speeds; decoded forward speed should match
    target = torch.empty(len(z), device=z.device).uniform_(float(SP_LO), float(SP_HI))
    xh = model.decode(z, (target - SP_MEAN) / SP_STD)
    got = xh[..., FWD].mean(1) * FWD_STD + FWD_MEAN
    return (((got - target) / FWD_STD) ** 2 * xh.shape[1]).mean()

## Train

In [ ]:
Xtr, Ctr = torch.tensor(data["train"]["X"]), torch.tensor(data["train"]["c"])
Xva, Cva = torch.tensor(data["val"]["X"]).to(device), torch.tensor(data["val"]["c"]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=PATIENCE // 3)

hist, best, best_state, wait = [], np.inf, None, 0
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    beta_t = BETA * min(1.0, epoch / max(1, KL_WARMUP_EPOCHS))
    model.train()
    perm = torch.randperm(len(Xtr))
    tr_rec = tr_kl = tr_ctl = 0.0
    for i in range(0, len(Xtr), BATCH_SIZE):
        b = perm[i:i + BATCH_SIZE]
        xb, cb = Xtr[b].to(device), Ctr[b].to(device)
        xh, mu, logvar = model(xb, cb)
        rec, kl = loss_terms(xh, xb, mu, logvar)
        loss = rec + beta_t * kl
        if CONTROL_WEIGHT > 0:
            z = mu + torch.randn_like(mu) * (0.5 * logvar).exp() if BETA > 0 else mu
            ctl = control_loss(z)
            loss = loss + CONTROL_WEIGHT * ctl
            tr_ctl += ctl.item() * len(xb)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_rec += rec.item() * len(xb); tr_kl += kl.item() * len(xb)
    model.eval()
    with torch.no_grad():
        xh, mu, logvar = model(Xva, Cva, sample=False)
        va_rec, va_kl = (v.item() for v in loss_terms(xh, Xva, mu, logvar))
        torch.manual_seed(1234)   # same random requests every epoch, so val loss is comparable
        va_ctl = control_loss(mu).item() if CONTROL_WEIGHT > 0 else 0.0
        torch.manual_seed(SEED + epoch)
    va_loss = va_rec + BETA * va_kl + CONTROL_WEIGHT * va_ctl
    sched.step(va_loss)
    hist.append(dict(epoch=epoch, beta=beta_t, train_rec=tr_rec / len(Xtr), train_kl=tr_kl / len(Xtr),
                     train_ctl=tr_ctl / len(Xtr), val_rec=va_rec, val_kl=va_kl, val_ctl=va_ctl, val_loss=va_loss))
    if va_loss < best - 1e-4 and epoch >= min(KL_WARMUP_EPOCHS, EPOCHS):
        best, best_state, wait = va_loss, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
    elif epoch >= KL_WARMUP_EPOCHS:
        wait += 1
    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch {epoch:4d} | train rec {tr_rec/len(Xtr):9.2f} kl {tr_kl/len(Xtr):7.2f} | "
              f"val rec {va_rec:9.2f} kl {va_kl:7.2f} ctl {va_ctl:7.2f} | β {beta_t:.2f} | {time.time()-t0:5.0f}s")
    if wait >= PATIENCE:
        print(f"early stop at epoch {epoch}")
        break

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state); model.eval()
hist = pd.DataFrame(hist); hist.to_csv(OUT_DIR / "history.csv", index=False)
torch.save(dict(state_dict=best_state, D=int(D), T=int(T), LATENT_DIM=LATENT_DIM, HIDDEN=HIDDEN, BETA=BETA,
                CONTROL_WEIGHT=CONTROL_WEIGHT, SP_MEAN=SP_MEAN, SP_STD=SP_STD, SPEED_COL=SPEED_COL, features=FEATS, pre_dir=str(PRE_DIR)),
           OUT_DIR / "model.pt")
print("saved", OUT_DIR / "model.pt")

fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
ax[2].plot(hist.epoch, hist.train_ctl, label="train"); ax[2].plot(hist.epoch, hist.val_ctl, label="val")
ax[2].set(yscale="symlog", xlabel="epoch", title="speed-control loss"); ax[2].legend()
ax[0].plot(hist.epoch, hist.train_rec, label="train"); ax[0].plot(hist.epoch, hist.val_rec, label="val")
ax[0].set(yscale="log", xlabel="epoch", title="reconstruction loss"); ax[0].legend()
ax[1].plot(hist.epoch, hist.train_kl, label="train"); ax[1].plot(hist.epoch, hist.val_kl, label="val")
ax[1].set(xlabel="epoch", title="KL divergence"); ax[1].legend()
plt.tight_layout(); plt.show()

## Helpers: features → skeleton, encode / decode

In [ ]:
def rot_xy(v, ang):
    c, s = np.cos(ang), np.sin(ang)
    shape = (len(ang),) + (1,) * (v.ndim - 2)
    c, s = c.reshape(shape), s.reshape(shape)
    out = v.copy()
    out[..., 0] = c * v[..., 0] - s * v[..., 1]
    out[..., 1] = s * v[..., 0] + c * v[..., 1]
    return out

def recover_joints(x, fps=FPS):
    T_ = len(x)
    th = np.r_[0.0, np.cumsum(x[:-1, 3]) / fps]
    v = rot_xy(np.c_[x[:, 1:3], np.zeros(T_)], th)[:, :2]
    root_xy = np.r_[np.zeros((1, 2)), np.cumsum(v[:-1], 0) / fps]
    local = rot_xy(x[:, 4:67].reshape(T_, 21, 3).astype(np.float64), th)
    local[..., :2] += root_xy[:, None]
    return np.concatenate([np.c_[root_xy, x[:, 0]][:, None], local], 1)

def local_pose(x):
    T_ = len(x)
    root = np.zeros((T_, 1, 3)); root[:, 0, 2] = x[:, 0]
    return np.concatenate([root, x[:, 4:67].reshape(T_, 21, 3)], 1)

def denorm(xn):
    x = xn * STD + MEAN
    x[..., CONTACT] = 1 / (1 + np.exp(-xn[..., CONTACT]))
    return x

@torch.no_grad()
def encode(X, C, bs=1024):
    out = [model.encode(torch.tensor(X[i:i + bs]).to(device), torch.tensor(C[i:i + bs]).to(device)) for i in range(0, len(X), bs)]
    return torch.cat([o[0] for o in out]).cpu().numpy(), torch.cat([o[1] for o in out]).cpu().numpy()

@torch.no_grad()
def decode(Z, C, bs=1024):
    return np.concatenate([model.decode(torch.tensor(Z[i:i + bs], dtype=torch.float32).to(device),
                                        torch.tensor(C[i:i + bs], dtype=torch.float32).to(device)).cpu().numpy()
                           for i in range(0, len(Z), bs)])

for s in data:
    data[s]["mu"], data[s]["logvar"] = encode(data[s]["X"], data[s]["c"])
    data[s]["recon"] = denorm(decode(data[s]["mu"], data[s]["c"]))
    data[s]["orig"] = data[s]["X"] * STD + MEAN
    np.savez(OUT_DIR / f"latents_{s}.npz", mu=data[s]["mu"], logvar=data[s]["logvar"],
             window_id=data[s]["meta"].window_id.values, speed=data[s]["speed"])

## 1. Reconstruction with the true speed (test people)

In [ ]:
te = data["test"]
xn_rec = (te["recon"] - MEAN) / STD
rj = np.stack([recover_joints(r) for r in te["recon"]])
res = dict(run=RUN_NAME,
           test_feature_rmse=float(np.sqrt(((xn_rec[..., REGULAR] - te["X"][..., REGULAR]) ** 2).mean())),
           test_pose_error_cm=float(np.mean([np.linalg.norm(local_pose(r) - local_pose(o), axis=-1).mean()
                                             for r, o in zip(te["recon"], te["orig"])]) * 100),
           test_full_error_cm=float(np.linalg.norm(rj - te["joints"], axis=-1).mean() * 100),
           test_contact_accuracy=float(((te["recon"][..., CONTACT] > 0.5) == (te["orig"][..., CONTACT] > 0.5)).mean()),
           speed_after_recon_R2=float(r2_score(te["speed"], te["recon"][..., FWD].mean(1))),
           speed_after_recon_MAE=float(mean_absolute_error(te["speed"], te["recon"][..., FWD].mean(1))))
for k, v in res.items():
    print(f"{k:24s} {v}")

## 2. Speed control: ask for a speed, measure what you get
Every test walk is encoded with its true speed, then decoded with each requested speed (same z, new speed input).
- **Control R²** / **MAE**: how close the measured speed is to the requested speed (1.0 / 0 = perfect)
- **Slope**: m/s obtained per m/s requested (1.0 = exact)
- **Spearman ρ** per walk: does faster requested always give faster walking?

Requested speeds cover the 2nd–98th percentile of the training speeds. Asking far outside that range is extrapolation.

In [ ]:
targets = np.linspace(*np.percentile(data["train"]["speed"], [2, 98]), N_TARGET_SPEEDS)
Z = te["mu"]
measured = np.stack([denorm(decode(Z, np.full(len(Z), to_c(t))))[..., FWD].mean(1) for t in targets], 1)   # (N, K)
req = np.broadcast_to(targets, measured.shape)
slope = np.polyfit(req.ravel(), measured.ravel(), 1)[0]
rho = np.array([spearmanr(targets, m)[0] for m in measured])
res.update(control_R2=float(r2_score(req.ravel(), measured.ravel())),
           control_MAE=float(np.abs(measured - req).mean()),
           control_slope=float(slope), control_mean_spearman=float(np.nanmean(rho)),
           control_target_min=float(targets[0]), control_target_max=float(targets[-1]),
           control_measured_min=float(measured[:, 0].mean()), control_measured_max=float(measured[:, -1].mean()))
print({k: round(v, 3) for k, v in res.items() if k.startswith("control")})
pd.DataFrame(measured, columns=[f"{t:.3f}" for t in targets]).assign(window_id=te["meta"].window_id.values) \
  .to_csv(OUT_DIR / "speed_control_test.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(targets, measured.T, color="tab:blue", alpha=.08)
ax[0].plot(targets, measured.mean(0), "k", lw=2, label="mean over test walks")
ax[0].plot(targets, targets, "r--", lw=1, label="perfect")
ax[0].set(xlabel="requested speed (m/s)", ylabel="measured speed of decoded walk (m/s)",
          title=f"speed control (R²={res['control_R2']:.2f}, slope={slope:.2f})"); ax[0].legend()
ax[1].hist(rho, bins=20); ax[1].set(xlabel="Spearman ρ per walk", title=f"monotonicity (mean ρ = {np.nanmean(rho):.2f})")
plt.tight_layout(); plt.show()

## 3. Does z still contain speed?
Ridge probe z → true speed (fit on train, scored on test people). **Low is good here**: speed should be carried by the speed input. A high value plus weak speed control means the decoder reads speed from z and ignores the input. Lowering `BETA` makes this worse; raising it helps.

In [ ]:
y = {s: data[s]["speed"] for s in data}
bestp = None
for a in [0.01, 0.1, 1, 10, 100, 1000]:
    p = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(data["train"]["mu"], y["train"])
    v = r2_score(y["val"], p.predict(data["val"]["mu"]))
    if bestp is None or v > bestp[0]:
        bestp = (v, p)
res["probe_z_to_speed_test_R2"] = float(r2_score(y["test"], bestp[1].predict(data["test"]["mu"])))
print(f"speed readable from z (test R²): {res['probe_z_to_speed_test_R2']:.3f}")

## 4. Stick figures
Top: one test walk decoded at slow / its own / fast speed. Bottom: brand-new walks from a random z at three speeds.

In [ ]:
def draw(ax, P, color, lw=1.5):
    for j, p in BONES:
        ax.plot([P[j, 0], P[p, 0]], [P[j, 2], P[p, 2]], color=color, lw=lw)

def show(ax, x, title):
    J3 = recover_joints(x)
    for k, t in enumerate(np.linspace(0, len(x) - 1, 6).astype(int)):
        draw(ax, J3[t], plt.cm.viridis(k / 5))
    ax.axhline(0, color="grey", lw=.6); ax.set_aspect("equal"); ax.set_title(title, fontsize=9)

i = len(Z) // 2
lo_t, hi_t = targets[1], targets[-2]
rng = np.random.default_rng(SEED)
z_new = rng.standard_normal((1, LATENT_DIM)).astype(np.float32)
rows = [(Z[i:i + 1], lo_t, "test walk, slow request"), (Z[i:i + 1], te["speed"][i], "test walk, its own speed"),
        (Z[i:i + 1], hi_t, "test walk, fast request"),
        (z_new, lo_t, "new walk, slow"), (z_new, float(np.median(data['train']['speed'])), "new walk, medium"), (z_new, hi_t, "new walk, fast")]
fig, axes = plt.subplots(len(rows), 1, figsize=(12, 2.4 * len(rows)))
for ax, (z, t, lab) in zip(axes, rows):
    x = denorm(decode(z, np.full(1, to_c(t))))[0]
    show(ax, x, f"{lab}: requested {t:.2f} m/s → measured {x[:, FWD].mean():.2f} m/s")
plt.tight_layout(); plt.show()

## 5. Compared with the VAE's speed direction (notebook 04)
Uses the results saved by the VAE run named in `VAE_RUN`, if it exists. A fair, detailed comparison (same speed targets, plausibility, cadence vs. stride) is part of the Phase-5 notebook.

In [ ]:
vae_file = PRE_DIR.parent / "models" / VAE_RUN / "results.json"
if vae_file.exists():
    v = json.loads(vae_file.read_text())
    comp = pd.DataFrame({
        "VAE + speed direction (04)": [v.get("test_pose_error_cm"), v.get("speed_after_recon_R2"),
                                        v.get("edit_speed_at_minus3"), v.get("edit_speed_at_plus3"), v.get("edit_mean_spearman")],
        "speed-conditioned VAE (05)": [res["test_pose_error_cm"], res["speed_after_recon_R2"],
                                        res["control_measured_min"], res["control_measured_max"], res["control_mean_spearman"]]},
        index=["pose error (cm)", "speed kept after reconstruction (R²)", "slowest edited speed (m/s)",
               "fastest edited speed (m/s)", "monotonicity (mean Spearman ρ)"])
    print(comp.round(3).to_string())
    print("\nVAE edit range = α from −3σ to +3σ along the direction; CVAE range = requested",
          f"{targets[0]:.2f}–{targets[-1]:.2f} m/s.")
else:
    print(f"No notebook-04 results at {vae_file} — train the VAE first (or set VAE_RUN) to see the comparison.")

## Save the results

In [ ]:
res.update(epochs_trained=int(hist.epoch.max()), best_val_loss=float(best), n_train=len(data["train"]["X"]),
           settings=dict(BETA=BETA, CONTROL_WEIGHT=CONTROL_WEIGHT, LATENT_DIM=LATENT_DIM, HIDDEN=HIDDEN, ROOT_WEIGHT=ROOT_WEIGHT, LR=LR,
                         BATCH_SIZE=BATCH_SIZE, SEED=SEED, SPEED_COL=SPEED_COL))
(OUT_DIR / "results.json").write_text(json.dumps(res, indent=2))
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items() if k != "settings"}, indent=2))
print("\nfiles in", OUT_DIR, ":", sorted(p.name for p in OUT_DIR.iterdir()))

## What to look for
- **Control R² high and slope near 1:** the CVAE controls speed well. This is the bar your speed direction is compared against.
- **Control slope much below 1, plus a high z → speed probe:** the decoder ignores the speed input (speed is in z instead). Raise `CONTROL_WEIGHT` (e.g. 5) or `BETA`.
- **For the write-up:** it's worth also training once with `CONTROL_WEIGHT = 0` and reporting it. It shows how a textbook conditional VAE behaves.
- **Reconstruction similar to the VAE** (pose error, speed kept), so the comparison is fair. A strong `CONTROL_WEIGHT` costs some reconstruction accuracy (a few cm of pose error); report that trade-off.
- Train 3 seeds (`SEED = 0, 1, 2`) like the VAE.